In [15]:
!pip install kagglehub opencv-python-headless

import kagglehub
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
import random

print("GPU:", tf.config.list_physical_devices('GPU'))

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [16]:
path = kagglehub.dataset_download("mohamedmustafa/real-life-violence-situations-dataset")
print("Downloaded to:", path)

Using Colab cache for faster access to the 'real-life-violence-situations-dataset' dataset.
Downloaded to: /kaggle/input/real-life-violence-situations-dataset


In [17]:
dataset_root = None

for root, dirs, files in os.walk(path):
    if "Violence" in dirs and "NonViolence" in dirs:
        dataset_root = root
        break

print("Using dataset root:", dataset_root)

Using dataset root: /kaggle/input/real-life-violence-situations-dataset/real life violence situations/Real Life Violence Dataset


In [18]:
data = []

for label_name, label in [("Violence", 1), ("NonViolence", 0)]:
    folder = os.path.join(dataset_root, label_name)

    files = os.listdir(folder)[:500]   # 🔥 LIMIT (speed)

    for file in files:
        if file.endswith(".mp4"):
            data.append((os.path.join(folder, file), label))

print("Total videos:", len(data))

Total videos: 979


In [19]:
random.shuffle(data)

split = int(0.8 * len(data))

train_data = data[:split]
val_data = data[split:]

print("Train:", len(train_data))
print("Validation:", len(val_data))

Train: 783
Validation: 196


In [20]:
def preprocess_frame(frame):
    small = cv2.resize(frame, (96, 96))   # 🔥 smaller
    frame = cv2.resize(small, (128, 128))

    frame = cv2.GaussianBlur(frame, (3,3), 0)

    return frame / 255.0

In [21]:
def load_video(path, max_frames=12):   # 🔥 reduced
    cap = cv2.VideoCapture(path)
    frames = []

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    step = max(1, total // max_frames)

    for i in range(0, total, step):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        ret, frame = cap.read()

        if not ret:
            break

        frame = preprocess_frame(frame)
        frames.append(frame)

        if len(frames) == max_frames:
            break

    cap.release()

    while len(frames) < max_frames:
        frames.append(frames[-1])

    return np.array(frames)

In [22]:
def data_generator(dataset, batch_size=4):
    while True:
        random.shuffle(dataset)

        for i in range(0, len(dataset), batch_size):
            batch = dataset[i:i+batch_size]

            X, y = [], []

            for video_path, label in batch:
                try:
                    frames = load_video(video_path)
                    X.append(frames)
                    y.append(label)
                except:
                    continue

            if len(X) > 0:
                yield np.array(X), np.array(y)

In [23]:
base_model = MobileNetV2(weights="imagenet", include_top=False, pooling="avg")

for layer in base_model.layers:
    layer.trainable = False

model = models.Sequential([
    layers.Input(shape=(12, 128, 128, 3)),  # 🔥 smaller input

    layers.TimeDistributed(base_model),

    layers.GlobalAveragePooling1D(),

    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

/tmp/ipykernel_734/2053541505.py:1: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = MobileNetV2(weights="imagenet", include_top=False, pooling="avg")


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed_2              │ (None, 12, 1280)       │     2,257,984 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │        81,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,340,033 (8.93 MB)

 Trainable params: 82,049 (320.50 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [24]:
batch_size = 4

train_gen = data_generator(train_data, batch_size)
val_gen = data_generator(val_data, batch_size)

steps_per_epoch = 100      # 🔥 FAST
validation_steps = 20

history = model.fit(
    train_gen,
    steps_per_epoch=steps_per_epoch,
    validation_data=val_gen,
    validation_steps=validation_steps,
    epochs=5   # 🔥 keep low for 2-hour limit
)

Epoch 1/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 479s 4s/step - accuracy: 0.5875 - loss: 0.7109 - val_accuracy: 0.7250 - val_loss: 0.5584
Epoch 2/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 371s 4s/step - accuracy: 0.6541 - loss: 0.5902 - val_accuracy: 0.8125 - val_loss: 0.4817
Epoch 3/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 370s 4s/step - accuracy: 0.7850 - loss: 0.4739 - val_accuracy: 0.8000 - val_loss: 0.4967
Epoch 4/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 315s 3s/step - accuracy: 0.8221 - loss: 0.4096 - val_accuracy: 0.8500 - val_loss: 0.4271
Epoch 5/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 377s 4s/step - accuracy: 0.8225 - loss: 0.3796 - val_accuracy: 0.8125 - val_loss: 0.4208


In [25]:
model.save("fight_detection_model.h5")

In [26]:
from google.colab import files
files.download("fight_detection_model.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>